In [1]:
import numpy as np
import pandas as pd
import psycopg2
import datetime

In [2]:
## insert proper credentials

conn = psycopg2.connect("dbname=kkdj20_test user=sebastian host=localhost password=123456")
#conn = psycopg2.connect("dbname=kraladies user=sebastian host=localhost password=123456")

cur = conn.cursor()

In [3]:
# use one or the other
# EVENT_NAME='KKM11'
EVENT_ID=9

cur.execute("SELECT * FROM baskets_event as be WHERE be.id = %s", (EVENT_ID,))
rn=cur.fetchone()
dashes="------------------------------------------"
print (dashes)
print (f"EVENT_ID {rn[0]}: {rn[1]} -- {rn[3]}")
print (dashes)


try:
    cur.execute('''SELECT sum(bi.price) AS cash
                        , users.username
                            FROM baskets_item AS bi 
                            JOIN baskets_vendor as bv 
                                ON bv.id=bi."vendorID_id"
                            JOIN baskets_basket as bb
                                ON bb.id=bi."basket_id"
                            JOIN auth_user as users
                                ON users.id=bi.created_by_id
                            WHERE bb."event_id"=%s
                            GROUP BY users.username
                            ORDER BY cash, users.username
                    ''',(EVENT_ID, ))
    res = cur.fetchall()
except Exception as e:
    print("database query failed. ", e)

#GROUP BY bi.created_by_id

totsum=0
for r in res:
    user=r[1]
    cash=r[0]
    totsum+=cash
    print (f"  {user:12s}: {cash:8.2f} Euro")
    #print (r)
print (dashes)
totstr="Total"
print (f"  {totstr:12s}: {totsum:8.2f} Euro")
print (dashes)

------------------------------------------
EVENT_ID 9: KKM15 -- 15. KKM
------------------------------------------
  gitte       :     9.50 Euro
  jasmin      :    26.00 Euro
  janine      :    65.00 Euro
  tini        :    85.00 Euro
  julie       :   256.75 Euro
  eugenie     :   742.50 Euro
  carina      :   944.20 Euro
  sarina      :  1105.30 Euro
  sarah       :  1700.60 Euro
  pina        :  1760.50 Euro
  tamara      :  2308.00 Euro
  katharina   :  2614.00 Euro
------------------------------------------
  Total       : 11617.35 Euro
------------------------------------------
